# 06_Agent 概览

预计阅读时间：10 分钟。

本章介绍一个单个 AI Coding Agent 的最小实现路径。它接收任务，调用大模型，根据需要使用工具，再把工具结果交还给模型继续推理。

和普通聊天应用相比，Agent 的关键区别是：它不只是生成文本，还能通过工具影响外部环境，并根据执行结果继续行动。


## ReAct 思想

现代 Agent 的一个重要思想来自 ReAct：Reasoning and Acting。它强调模型不是一次性给出答案，而是在推理和行动之间循环：

```text
思考下一步 -> 调用工具 -> 观察结果 -> 继续思考
```

本章的 Coding Agent 可以看作 ReAct 思想在编程任务中的一个最小版本：模型决定是否调用工具，程序执行工具并返回结果，模型再根据结果继续完成任务。

这里不展开论文细节，只需要记住一点：Agent 是“推理 + 工具行动 + 观察结果”的循环系统。


## 本章目标

本章不追求一次性实现完整 Claude Code，而是围绕单个 Agent 逐步理解几个关键机制：

- Agent Loop：模型、工具和执行结果如何形成循环；
- Bash Tool：模型如何通过工具操作真实环境；
- Skill：如何按需加载可复用经验；
- MCP：如何用标准方式接入外部工具；
- Agent Harness：完整 Agent 系统还需要哪些运行框架能力。

学习重点是理解结构，而不是堆叠功能。


## 章节路线

| 小节 | Notebook | 脚本 | 重点 |
| --- | --- | --- | --- |
| 01 | 本节 | `01_agent.py` | 章节概览和 ReAct 思想 |
| 02 | [02_你的第一个Agent.ipynb](02_你的第一个Agent.ipynb) | `02_agent.py` | 一个循环和一个 Bash 工具 |
| 03 | [03_AgentSkill.ipynb](03_AgentSkill.ipynb) | `03_agent.py` | 使用 `SKILL.md` 按需加载经验 |
| 04 | [04_AgentMCP.ipynb](04_AgentMCP.ipynb) | `04_agent.py` | 通过 MCP mock server 接入外部工具 |
| 05 | [05_Agent_Harness.ipynb](05_Agent_Harness.ipynb) | - | 按 S01-S20 总结 Agent Harness |

其中 `01_agent.py` 是最原始的命令行 Agent Loop 示例；`02_agent.py` 在此基础上增加了 history 保存，更适合作为第一个完整运行示例。


## 文件结构

本章当前主要文件包括：

```text
books/06_Agent/
├── 01_概览.ipynb
├── 02_你的第一个Agent.ipynb
├── 03_AgentSkill.ipynb
├── 04_AgentMCP.ipynb
├── 05_Agent_Harness.ipynb
├── 01_agent.py
├── 02_agent.py
├── 03_agent.py
├── 04_agent.py
├── figures/
└── skills/
```

`.env`、`.memory/`、`.tasks/` 等运行时文件不进入 Git 记录。


## 环境检查

下面的检查不会调用大模型，也不会运行交互式 Agent，只确认本章需要的文件和环境变量是否存在。


In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

chapter_dir = Path.cwd()
if not (chapter_dir / "01_概览.ipynb").exists():
    chapter_dir = Path("books/06_Agent")

if load_dotenv is not None:
    for env_path in [chapter_dir / ".env", Path(".env")]:
        if env_path.exists():
            load_dotenv(env_path, override=True)

required_files = [
    "02_agent.py",
    "03_agent.py",
    "04_agent.py",
    "skills/python-docstring/SKILL.md",
]

for name in required_files:
    print(f"{name}:", "存在" if (chapter_dir / name).exists() else "缺失")

print("ANTHROPIC_AUTH_TOKEN:", "已设置" if os.getenv("ANTHROPIC_AUTH_TOKEN") else "未设置")
print("ANTHROPIC_BASE_URL:", os.getenv("ANTHROPIC_BASE_URL") or "未设置")
print("MODEL_ID:", os.getenv("MODEL_ID") or "未设置")


## 如何运行

如果要体验命令行 Agent，可以在本章目录下运行：

```bash
python3 02_agent.py
```

如果要体验 Skill：

```bash
python3 03_agent.py
```

如果要体验 MCP mock 工具：

```bash
python3 04_agent.py
```

运行前需要在 `books/06_Agent/.env` 中配置模型接口信息。本课程不把 `.env` 提交到 Git。


## 小结

本章从最小 Agent Loop 出发，逐步加入 Skill 和 MCP，最后用 Agent Harness 总结完整系统还需要的能力。

可以把本章主线理解为：

```text
Agent Loop -> Skill -> MCP -> Harness
```

后续如果继续扩展，可以从 Todo、Memory、System Prompt、Task System 和 Worktree Isolation 等 Harness 机制开始。
